# 실험 03: Structured Output (API 레벨 구조화 강제)

## 1. 실험 제목과 목표
- **제목**: 최신 트렌드 `.with_structured_output()` 마스터하기
- **목표**: 텍스트를 나중에 파싱하는 구식(Output Parser) 방식을 버리고, 처음부터 모델이 JSON/Pydantic 객체 자체를 뱉어내도록 강제하는 가장 강력하고 안정적인 최신 방법을 배웁니다. 더불어 정보 추출(Information Extraction) 실무 패턴도 실습합니다.

## 2. 실험 개요
1. **비교 실험 1**: 기존 `Parser` 체인 조립 방식 vs `with_structured_output()` 래핑 방식 비교
2. **실무 실험 2**: 긴 뉴스 기사에서 여러 인물(Entities)의 정보만 쏙쏙 뽑아내는 Extraction 시연
3. **실패/주의 케이스**: 프롬프트가 너무 부실할 때 모델이 빈 값을 채워넣는(환각) 현상 관찰

## 3. 라이브러리 import

In [3]:
import os
from dotenv import load_dotenv
from typing import List, Optional
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

## 4. 환경 변수 로드 및 모델 준비

In [4]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

## 5. 비교 실험 1: 기존 파서 방식 vs Structured Output
02번 실험에서 했던 코드가 얼마나 드라마틱하게 짧아지는지 확인합니다.

In [ ]:
class AnimalInfo(BaseModel):
    name: str = Field(description="동물의 이름")
    lifespan: int = Field(description="평균 수명 (연도)")

print("=== [1. 기존 Output Parser 방식 복습] ===")
old_parser = PydanticOutputParser(pydantic_object=AnimalInfo)
old_prompt = PromptTemplate.from_template("{animal}에 대해 알려줘.\n{format_instructions}")
old_chain = old_prompt | llm | old_parser
print(old_chain.invoke({"animal": "코끼리", "format_instructions": old_parser.get_format_instructions()}))

print("\n=== [2. 최신 Structured Output 방식] ===")
# 모델 자체에 Pydantic 스키마를 '바인딩(Binding)' 해버립니다.
structured_llm = llm.with_structured_output(AnimalInfo)

# 프롬프트에 아무런 포맷 지시를 할 필요가 없습니다!
new_prompt = PromptTemplate.from_template("{animal}에 대해 알려줘.")

# 파서를 파이프라인에 넣을 필요도 없습니다!
new_chain = new_prompt | structured_llm
res = new_chain.invoke({"animal": "코끼리"})

print("데이터 타입:", type(res))
print("결과 객체:", res)
print("\n-> 결과: 코드가 압도적으로 간결해졌으며, 내부적으로 OpenAI의 Tool Calling(또는 JSON Mode) API를 직접 타기 때문에 파싱 에러 확률이 거의 제로에 수렴합니다.")

=== [1. 기존 Output Parser 방식 복습] ===
name='코끼리' lifespan=60

=== [2. 최신 Structured Output 방식] ===
데이터 타입: <class '__main__.AnimalInfo'>
결과 객체: name='코끼리' lifespan=60

-> 결과: 코드가 압도적으로 간결해졌으며, 내부적으로 OpenAI의 Tool Calling(또는 JSON Mode) API를 직접 타기 때문에 파싱 에러 확률이 거의 제로에 수렴합니다.


## 6. 실무 실험 2: 정보 추출 (Information Extraction)
뉴스 기사나 복잡한 텍스트에서 여러 명의 인물 정보(이름, 소속, 특징)를 배열(List) 형태로 쏙쏙 빼내는 실무 패턴입니다.

In [6]:
# 단일 인물 스키마
class Person(BaseModel):
    name: str = Field(description="인물의 이름")
    organization: Optional[str] = Field(description="소속된 회사나 조직. 언급이 없으면 None", default=None)
    role: str = Field(description="직책이나 역할")

# 인물들의 리스트를 가지는 '최상위 스키마'
class ExtractionResult(BaseModel):
    people: List[Person] = Field(description="텍스트에서 언급된 인물들의 목록")
    summary: str = Field(description="기사 전체의 1줄 요약")

# 스키마 바인딩
extractor = llm.with_structured_output(ExtractionResult)

news_article = """
오늘 애플의 CEO 팀 쿡은 신제품 발표회에서 극찬을 받았습니다. 
이 행사에는 구글의 순다르 피차이도 몰래 참석하여 메모를 하는 모습이 포착되었죠. 
한 무명 개발자인 김코딩 씨는 현장에서 환호성을 지르다가 퇴장당했습니다.
"""

print("[기사에서 정보 추출 시작]")
# 그냥 문자열을 모델에 통째로 던집니다.
extracted_data = extractor.invoke(news_article)

print("기사 요약:", extracted_data.summary)
print("\n[추출된 인물 목록]")
for p in extracted_data.people:
    print(f"이름: {p.name} | 소속: {p.organization} | 역할: {p.role}")

[기사에서 정보 추출 시작]
기사 요약: 오늘 애플의 신제품 발표회에서 CEO 팀 쿡이 극찬을 받았고 구글 CEO 순다르 피차이도 몰래 참석하여 메모를 하였으며, 한 무명 개발자 김코딩 씨는 퇴장당하는 사건이 있었다.

[추출된 인물 목록]
이름: 팀 쿡 | 소속: 애플 | 역할: CEO
이름: 순다르 피차이 | 소속: 구글 | 역할: CEO
이름: 김코딩 | 소속: None | 역할: 개발자


## 7. 실패/주의 케이스: 부실한 텍스트로 인한 강제 생성(환각)
Structured Output은 스키마에 정의된 값이 '반드시' 있어야 한다고 강제하므로, 텍스트에 해당 정보가 없으면 모델이 지어내버릴 수 있습니다.

In [7]:
print("[정보가 없는 텍스트 주입]")
empty_text = "오늘은 참 날씨가 좋네요."

weird_result = extractor.invoke(empty_text)
print("요약:", weird_result.summary)
print("추출 인물:", weird_result.people)

print("\n-> 주의: 텍스트에 인물이 전혀 없지만, 모델은 '반드시 채워야 한다'는 압박감에 환각(Hallucination)을 일으킬 수 있습니다.")
print("   이를 방지하려면 스키마에 Optional을 잘 선언하거나, 프롬프트에 '정보가 없으면 빈 리스트를 반환하라'고 명시해야 합니다.")

[정보가 없는 텍스트 주입]
요약: 날씨가 좋다.
추출 인물: []

-> 주의: 텍스트에 인물이 전혀 없지만, 모델은 '반드시 채워야 한다'는 압박감에 환각(Hallucination)을 일으킬 수 있습니다.
   이를 방지하려면 스키마에 Optional을 잘 선언하거나, 프롬프트에 '정보가 없으면 빈 리스트를 반환하라'고 명시해야 합니다.


## 8. 결과 해석

1. **패러다임의 변화**: `OutputParser` 시대에는 프롬프트를 이쁘게 깎아서 에러를 막는 데 집중했다면, `Structured Output` 시대에는 데이터 설계도(Pydantic Schema)를 정교하게 깎는 데 집중하게 됩니다.
2. **API 네이티브 기능**: OpenAI, Anthropic 등 최신 모델들은 API 단에서 이런 구조화된 응답을 보장(Tool Calling/Function Calling 활용)하므로, 개발자는 편하게 파이썬 객체만 다루면 됩니다.
3. **강제력의 양날의 검**: 틀을 너무 꽉 짜두면, 모델이 틀을 채우기 위해 없는 정보를 지어낼 수 있으므로 `Optional` 활용이 필수적입니다.

## 9. Notion 실험 로그

### 발견한 문제점 (또는 차이점)
* `PydanticOutputParser`를 쓸 때보다 프롬프트가 사라지고 코드도 절반으로 줄어들었음.
* 텍스트에서 5개의 항목을 추출하라고 할 때 정규식(Regex)을 쓸 필요 없이 스키마만 던져주면 알아서 배열로 만들어줌. 정말 혁명적임.

### 다음 개선 방향
* `with_structured_output`이 만능은 아님. 오픈소스 LLM 등 기능 미지원 모델을 쓸 때나 아주 복잡한 엣지 케이스에서는 결국 파서 에러가 날 수 있음.
* 따라서 에러가 났을 때 LLM에게 **"너 양식 틀렸어. 다시 해와"**라고 스스로 고치게 만드는 `Auto-fixing` 기법을 배워 서비스 안정성을 99.9%로 올려야 함.